RAG EVALUATION USING DEEP EVALS

In [1]:
CORPUS = [
    {"title": "Tokens", "content": (
        "Large language models split text into tokens, common character sequences roughly 4 "
        "characters or three-quarters of a word long. Both the prompt and the output are counted "
        "in tokens, and pricing and context limits are measured in tokens.")},
    {"title": "Embeddings", "content": (
        "An embedding is a fixed-length vector that represents the meaning of a piece of text. "
        "Texts with similar meaning have vectors that are close together, usually measured by "
        "cosine similarity, which is what lets a system do semantic search.")},
    {"title": "Retrieval-Augmented Generation", "content": (
        "RAG grounds an LLM's answer in external documents. At query time the system retrieves "
        "the most relevant chunks and passes them to the model as context, which reduces "
        "hallucination and lets you update knowledge without retraining.")},
    {"title": "Hallucination", "content": (
        "A hallucination is fluent, confident text that is factually wrong or unsupported by its "
        "sources. It happens because the model predicts likely text rather than looking facts up. "
        "Grounding answers in retrieved context is the main defense.")},
    {"title": "AI agents", "content": (
        "An AI agent is an LLM given a goal, tools, and a loop: plan, call a tool, observe the "
        "result, decide the next step, until the task is done.")},
]


RETRIVER

In [2]:
def retrieve(query: str, k: int = 2) -> list[str]:
    """Naive retriever: rank docs by keyword overlap with the query. Good enough to teach evals."""
    q_words = set(query.lower().split())
    scored = []
    for doc in CORPUS:
        doc_words = set((doc["title"] + " " + doc["content"]).lower().split())
        overlap = len(q_words & doc_words)
        scored.append((overlap, doc))
    scored.sort(key=lambda x: x[0], reverse=True)
    return [doc["content"] for _, doc in scored[:k]]

RAG PIPELINE

In [3]:
from groq import Groq

groq_client = Groq()


RAG_SYSTEM = (
    "Answer ONLY using the provided context. If the context doesn't contain the answer, say you "
    "don't have that information. Be concise (1-2 sentences)."
)



def answer_with_groq(question: str, passages: list[str]) -> str:
    context = "\n\n".join(passages)
    prompt = f"Context:\n{context}\n\nQuestion: {question}\n\nAnswer:"
    resp = groq_client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[{"role": "system", "content": RAG_SYSTEM}, {"role": "user", "content": prompt}],
        temperature=0,
    )
    return resp.choices[0].message.content.strip()

In [4]:
QUESTIONS = [
    {"input": "What is a token in an LLM?",
     "expected_output": "A token is a common character sequence, roughly 4 characters, used to measure both prompt and output length."},
    {"input": "How does RAG reduce hallucination?",
     "expected_output": "RAG retrieves relevant chunks and passes them as context, grounding the answer in real documents instead of only the model's training."},
    {"input": "How do I containerize a model for deployment with Docker?"},  # not in our tiny KB
]


In [5]:

rag_rows = []
for q in QUESTIONS:
    passages = retrieve(q["input"])
    answer = answer_with_groq(q["input"], passages)
    rag_rows.append({**q, "actual_output": answer, "retrieval_context": passages})
    print("Q:", q["input"])
    print("A:", answer)
    print("-" * 80)

Q: What is a token in an LLM?
A: I don't have that information.
--------------------------------------------------------------------------------
Q: How does RAG reduce hallucination?
A: RAG reduces hallucination by grounding the model’s response in retrieved external documents—providing relevant chunks as context so the answer is based on factual sources rather than generated from memory alone.
--------------------------------------------------------------------------------
Q: How do I containerize a model for deployment with Docker?
A: I don't have that information.
--------------------------------------------------------------------------------


CREATING LLM TEST CASES

In [6]:
from deepeval.test_case import LLMTestCase

test_cases = [
    LLMTestCase(
        input=r["input"],
        actual_output=r["actual_output"],
        expected_output=r.get("expected_output"),
        retrieval_context=r["retrieval_context"],
    )
    for r in rag_rows
]

PERFORMING EVALS

In [7]:
from deepeval.models import LocalModel
import os
judge = LocalModel(
    model="llama-3.1-8b-instant",
    api_key=os.environ["GROQ_API_KEY"],
    base_url="https://api.groq.com/openai/v1",
    temperature=0,
)
print("Judge ready:", judge.get_model_name())

Judge ready: llama-3.1-8b-instant (Local Model)


In [8]:
from deepeval.metrics import (
    AnswerRelevancyMetric,
    ContextualPrecisionMetric,
    ContextualRecallMetric,
    ContextualRelevancyMetric,
    FaithfulnessMetric,
)

metrics = [
    FaithfulnessMetric(model=judge),
    AnswerRelevancyMetric(model=judge),
    ContextualRelevancyMetric(model=judge),
    ContextualPrecisionMetric(model=judge),
    ContextualRecallMetric(model=judge),
]

In [ ]:
from deepeval import evaluate
from deepeval.evaluate.configs import AsyncConfig, DisplayConfig, ErrorConfig

results = evaluate(
    test_cases=test_cases,
    metrics=metrics,
    async_config=AsyncConfig(max_concurrent=1, throttle_value=2.0),  # ~41 calls -- spread the burst
    display_config=DisplayConfig(print_results=True),
    error_config=ErrorConfig(ignore_errors=True, skip_on_missing_params=True),
)

✨ You're running DeepEval's latest Faithfulness Metric! (using llama-3.1-8b-instant (Local Model), strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Answer Relevancy Metric! (using llama-3.1-8b-instant (Local Model), 
strict=False, async_mode=True)...

✨ You're running DeepEval's latest Contextual Relevancy Metric! (using llama-3.1-8b-instant (Local Model), 
strict=False, async_mode=True)...

✨ You're running DeepEval's latest Contextual Precision Metric! (using llama-3.1-8b-instant (Local Model), 
strict=False, async_mode=True)...

✨ You're running DeepEval's latest Contextual Recall Metric! (using llama-3.1-8b-instant (Local Model), 
strict=False, async_mode=True)...

d:\Gen AI Projects\DeepEvals\.venv\Lib\site-packages\rich\live.py:260: UserWarning: install "ipywidgets" for 
Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')